In [ ]:
import sys; sys.path.append('..'); sys.path.append('../../../');  sys.path.append('../../../gmsh')

In [ ]:
import experiment_helper
import igl
from periodic_simulation_setup import *
import json


In [ ]:
import importlib
importlib.reload(pattern_generator_using_gmsh)

In [ ]:
# avg_len = 0.1
# h = 5
# radius = 1
# boundary_vxs, boundary_lines, base_arc_points, arc_edges = pattern_generator_using_gmsh.get_heart(h, 1.5, 1.5, avg_len, avg_len)        

In [ ]:
# visualization.plot_line_segments(base_arc_points.tolist() + boundary_vxs.tolist(), (arc_edges - 1).tolist() + (boundary_lines + len(base_arc_points) - 1).tolist())

In [ ]:

# avg_len = 0.1
# h = 5
# ipu, m, marker = pattern_generator_using_gmsh.get_heart(h, 1, 1, avg_len, avg_len)   


avg_len = 0.2
h = 5
w = 5
ipu, m, marker = pattern_generator_using_gmsh.get_aeromorph(w, h, 2.3, 0.5, avg_len, avg_len)

In [ ]:
# avg_len = 0.1
# h = 5
# radius = 2
# ipu, m, marker = pattern_generator_using_gmsh.get_circular_arc(h, radius, 0.1, 0.7, avg_len, avg_len)        

In [ ]:
visualization.plot_2d_mesh(m, pointList = marker, width = 5, height = 5)

In [ ]:
finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= -1)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= -1)

m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= -1)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= -1)

m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= -1)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= -1)


fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)


In [ ]:
fuse_boundary = True

In [ ]:
# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            fusedVtx[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            fusedVtx[i] = True    

In [ ]:

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)


viewer.showWireframe(True)

viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.05
scale_factor_pressure = 0.01

In [ ]:
allowBending = True
useTFT = True
disableFusedRegionTFT = False

In [ ]:
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
def cb(i):
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
ipu.visualizationTilePower = 0

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
benchmark.report()

In [ ]:
isheet = ipu.sheet



In [ ]:
sheet_viewer = TriMeshViewer(isheet, width=768, height=640)


sheet_viewer.showWireframe(True)

sheet_viewer.show()

In [ ]:
def sheet_cb(i):
    sheet_viewer.update(scalarField=utils.getStrains(isheet)[:, 0])

In [ ]:

# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

isheet.setUseTensionFieldEnergy(useTFT)
isheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    isheet.disableFusedRegionTensionFieldTheory(False)
isheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(isheet, fixedVars, opts, callback=sheet_cb, hessianShift = hessianShift)
sheet_viewer.update(scalarField=utils.getStrains(isheet)[:, 0])
benchmark.report()

In [ ]:
ipu.get_kappa(), ipu.get_alpha()

In [ ]:
curr_vars = ipu.getVars()
curr_vars[-1] =0
curr_vars[-2] = 0.1
ipu.setVars(curr_vars)

In [ ]:
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(True)
az_viewer.show()

In [ ]:
az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
def az_cb(i):
    az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, useTFT, disableFusedRegionTFT)
if not allowBending:
    fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=az_cb, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
az_ipu.ipu.get_alpha()

In [ ]:
az_ipu.ipu.get_kappa()

In [ ]:
curr_vars = az_ipu.getVars()
curr_vars[-1] =np.pi / 2
curr_vars[-2] = 0.
az_ipu.setVars(curr_vars)

In [ ]:
points = visualize_average_deformation_gradient(ipu, 100)


In [ ]:

name = "three_star_hex"
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  


In [ ]:
variable = 0
low_pressure_tag = "low_pressure"
high_pressure_tag = "high_pressure"


In [ ]:
np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)
print("Solved using stiffness shift: ", stiffness_shift)

In [ ]:

np.save("{}/experiment_parameters.npy".format(result_folder), np.array([stiffness_pressure, scale_factor_pressure, allowBending, disableFusedRegionTFT, useTFT]))

np.save("{}/{}_strain_values_{}_{}.npy".format(result_folder, high_pressure_tag, name, variable), utils.getStrains(az_ipu.ipu.sheet)[:, 0])


render = viewer.offscreenRenderer(1000, 1000)
render.render()
render.save("{}/{}_render_{}_{}.png".format(result_folder, high_pressure_tag, name, variable))
np.save("{}/{}_dofs_{}_{}.npy".format(result_folder, high_pressure_tag, name, variable), az_ipu.getVars())

In [ ]:
np.save("{}/scale_factors_{}_{}.npy".format(result_folder, name, variable), get_deformation_scale_factors(az_ipu.ipu))
np.save("{}/kappa_{}_{}.npy".format(result_folder, name, variable), az_ipu.getVars()[-2])

np.save("{}/average_deformation_gradient_matrix_{}_{}.npy".format(result_folder, name, variable), get_deformation_matrix(az_ipu.ipu))

np.save("{}/{}_strain_values_{}_{}.npy".format(result_folder, low_pressure_tag, name, variable), utils.getStrains(az_ipu.ipu.sheet)[:, 0])

np.save("{}/{}_dofs_{}_{}.npy".format(result_folder, low_pressure_tag, name, variable), az_ipu.getVars())

In [ ]:
import periodic_simulation_setup
importlib.reload(periodic_simulation_setup)
from periodic_simulation_setup import *

In [ ]:

stiffness_shift = 1e-15
success = False
for i in range(15):
    try:
        stiffness_values, sampled_alphas, stiffness_coefficient = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = stiffness_shift, fixedVars = [], filename = "{}/stiffness_{}_{}.png".format(result_folder, name, variable))
        np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
        np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)
        np.save("{}/stiffness_coefficient_{}_{}.npy".format(result_folder, name, variable), stiffness_coefficient)
        print("Solved using stiffness shift: ", stiffness_shift)
        success = True
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
        stiffness_shift *= 10
if (not success):
    print("Failed to solve stiffness!")

In [ ]:
max(az_ipu.ipu.sheet.getVars().reshape((int(az_ipu.ipu.sheet.numVars() / 3), 3))[:, 2])

In [ ]:
points = visualize_average_deformation_gradient(ipu, 100, filename = "{}/average_deformation_gradient_{}_{}.png".format(result_folder, name, variable), plot_min_r=0, plot_max_r=1)
